# Elevación del segmento ST: cómo reconocerla en el ECG

**Para qué sirve.** En la cadena **ICAT³** (Identificación, Confirmación, Activación, Tratamiento, Traslado, reTorno)
del programa nacional IAM_MX y en el **Código Infarto** del IMSS, el ECG se toma en **menos de 10 minutos** y se
lee de inmediato. Uno de los principales retrasos en México es el **tiempo al diagnóstico**: obtener **e interpretar**
el ECG. Este cuaderno le enseña a reconocer la elevación del ST que activa la reperfusión.

**Cómo usarlo:** menú **Entorno de ejecución → Ejecutar todas**. Espere 1–2 minutos y lea hacia abajo.

---
**Autor:** Víctor Junior Godínez-Vázquez · Material docente · Texto CC BY 4.0 · Código MIT
**Datos:** PTB-XL v1.0.3 (PhysioNet, CC BY 4.0). Wagner P, et al. *Sci Data* 2020;7:154.
**Advertencia:** material exclusivamente docente; no usar para decisiones clínicas.
*Elaborado con asistencia de IA (Claude, Anthropic); el autor revisó el contenido y es responsable de él.*

In [ ]:
#@title ⚙️ Paso 0 · Preparar el entorno (ejecute y espere ~30 s)
import os, sys, importlib, warnings
warnings.filterwarnings('ignore')
try:
    import wfdb
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'wfdb'], check=True)
_FUENTES = {'ecg_st_core.py': '# -*- coding: utf-8 -*-\n"""\nMotor del entorno virtual "Reconocimiento de elevación del segmento ST".\nFase 1: solo elevación del ST (sin imitadores, sin otros diagnósticos).\n\nDatos: PTB-XL v1.0.3 (PhysioNet), ECG de 12 derivaciones, 10 s, 500 Hz, en mV.\nCriterio: Quinta Definición Universal del Infarto (ESC/ACC/AHA/WHF, 2026): elevación\nNUEVA del ST medida en el punto J, en >= 2 derivaciones contiguas, en ausencia de\nbloqueo de rama y/o hipertrofia ventricular izquierda:\n  >= 1 mm en todas las derivaciones excepto V2-V3;\n  V2-V3: >= 2 mm hombres >= 40 años; >= 2.5 mm hombres < 40 años; >= 1.5 mm mujeres.\n\nAutor: Víctor Junior Godínez-Vázquez. Código: licencia MIT.\nMaterial docente; no usar para decisiones clínicas.\n"""\nimport ast\nimport os\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import signal as sps\nimport matplotlib.pyplot as plt\nfrom matplotlib.collections import LineCollection\n\nLEADS = [\'I\', \'II\', \'III\', \'aVR\', \'aVL\', \'aVF\',\n         \'V1\', \'V2\', \'V3\', \'V4\', \'V5\', \'V6\']\nPTBXL_DIR = \'ptb-xl/1.0.3/\'\nPTBXL_URL = \'https://physionet.org/files/ptb-xl/1.0.3/\'\n\n# Territorios (grupos de derivaciones contiguas) para la fase 1.\n# aVR se excluye a propósito: su elevación aislada es otro tema (fase posterior).\nTERRITORIOS = {\n    \'Inferior\': [\'II\', \'III\', \'aVF\'],\n    \'Lateral\': [\'I\', \'aVL\', \'V5\', \'V6\'],\n    \'Anterior\': [\'V1\', \'V2\', \'V3\', \'V4\'],\n}\nPRECORDIALES = [\'V1\', \'V2\', \'V3\', \'V4\', \'V5\', \'V6\']\n\n# Colores del papel de ECG\n_MINOR = \'#d9d9d9\'\n_MAJOR = \'#9a9a9a\'\n_TRACE = \'#111111\'\n\n_CACHE = {}\n\n\n# ----------------------------------------------------------------------------\n# Carga de datos\n# ----------------------------------------------------------------------------\ndef cargar_metadatos():\n    """Tabla de PTB-XL (una fila por ECG) y catálogo de declaraciones SCP."""\n    db = pd.read_csv(PTBXL_URL + \'ptbxl_database.csv\', index_col=\'ecg_id\')\n    db[\'scp_codes\'] = db[\'scp_codes\'].apply(ast.literal_eval)\n    scp = pd.read_csv(PTBXL_URL + \'scp_statements.csv\', index_col=0)\n    return db, scp\n\n\ndef cargar_registro(filename_hr):\n    """Devuelve (señal [n_muestras x 12] en mV en el orden de LEADS, fs)."""\n    if filename_hr in _CACHE:\n        return _CACHE[filename_hr]\n    import wfdb\n    carpeta, nombre = os.path.split(filename_hr)\n    rec = wfdb.rdrecord(nombre, pn_dir=PTBXL_DIR + carpeta)\n    nombres = [n.upper() for n in rec.sig_name]\n    orden = [nombres.index(l.upper()) for l in LEADS]\n    sig = np.asarray(rec.p_signal, dtype=float)[:, orden]\n    sig = np.nan_to_num(sig)\n    _CACHE[filename_hr] = (sig, rec.fs)\n    return sig, rec.fs\n\n\n# ----------------------------------------------------------------------------\n# Procesamiento\n# ----------------------------------------------------------------------------\ndef filtrar(sig, fs):\n    """Filtrado de fase cero que preserva el segmento ST.\n    Paso alto 0.05 Hz (recomendación AHA para no deformar el ST),\n    paso bajo 40 Hz y muesca a 50 Hz (red eléctrica de Alemania, origen de PTB-XL).\n    """\n    hp = sps.butter(2, 0.05, btype=\'highpass\', fs=fs, output=\'sos\')\n    lp = sps.butter(4, 40, btype=\'lowpass\', fs=fs, output=\'sos\')\n    y = sps.sosfiltfilt(hp, sig, axis=0)\n    y = sps.sosfiltfilt(lp, y, axis=0)\n    b, a = sps.iirnotch(50, 30, fs=fs)\n    y = sps.filtfilt(b, a, y, axis=0)\n    return y\n\n\ndef detectar_R(sig, fs):\n    """Picos de los complejos QRS a partir de la energía de la pendiente en las 12 derivaciones."""\n    bp = sps.butter(2, [5, 20], btype=\'bandpass\', fs=fs, output=\'sos\')\n    x = sps.sosfiltfilt(bp, sig, axis=0)\n    env = np.sum(np.abs(np.diff(x, axis=0)), axis=1)\n    w = max(1, int(0.08 * fs))\n    env = np.convolve(env, np.ones(w) / w, mode=\'same\')\n    umbral = 0.35 * np.percentile(env, 99)\n    picos, _ = sps.find_peaks(env, height=umbral, distance=int(0.30 * fs))\n    # Refinar: máximo de la señal absoluta en derivación con mayor amplitud\n    lead = int(np.argmax(np.ptp(sig, axis=0)))\n    r = []\n    for p in picos:\n        a, b = max(0, p - int(0.06 * fs)), min(len(sig), p + int(0.06 * fs))\n        r.append(a + int(np.argmax(np.abs(sig[a:b, lead] - np.median(sig[a:b, lead])))))\n    return np.array(sorted(set(r)), dtype=int)\n\n\ndef latido_mediano(sig, fs, pre=0.30, post=0.50):\n    """Latido mediano por derivación (reduce ruido; es lo que miden los electrocardiógrafos)."""\n    r = detectar_R(sig, fs)\n    a, b = int(pre * fs), int(post * fs)\n    ventanas = [sig[p - a:p + b] for p in r if p - a >= 0 and p + b <= len(sig)]\n    if len(ventanas) < 3:\n        return None, len(ventanas)\n    return np.median(np.stack(ventanas), axis=0), len(ventanas)\n\n\ndef puntos_fiduciales(med, fs, pre=0.30):\n    """Inicio del QRS y punto J (fin del QRS) en el latido mediano."""\n    g = np.sum(np.abs(np.gradient(med, axis=0)), axis=1)\n    w = max(1, int(0.01 * fs))\n    g = np.convolve(g, np.ones(w) / w, mode=\'same\')\n    r = int(pre * fs)\n    zona = g[r - int(0.08 * fs): r + int(0.08 * fs)]\n    umbral = 0.12 * zona.max()\n    sostener = int(0.012 * fs)\n    # Inicio: hacia atrás desde R\n    onset = r - int(0.04 * fs)\n    for i in range(r, r - int(0.15 * fs), -1):\n        if np.all(g[i - sostener:i] < umbral):\n            onset = i\n            break\n    # Punto J: hacia adelante desde R\n    j = r + int(0.06 * fs)\n    for i in range(r + int(0.02 * fs), r + int(0.16 * fs)):\n        if np.all(g[i:i + sostener] < umbral):\n            j = i\n            break\n    return onset, j\n\n\ndef analizar(sig, fs, edad=None, sexo=None, ajuste_J_ms=0):\n    """Mide el desnivel del ST en el punto J en cada derivación.\n\n    sexo: \'H\' (hombre) o \'M\' (mujer).  ajuste_J_ms: corrección manual del instructor.\n    Devuelve dict con latido mediano, puntos fiduciales y ST (mm) por derivación.\n    """\n    y = filtrar(sig, fs)\n    med, n = latido_mediano(y, fs)\n    if med is None:\n        return None\n    onset, j = puntos_fiduciales(med, fs)\n    j = j + int(round(ajuste_J_ms * fs / 1000))\n    b0, b1 = onset - int(0.025 * fs), onset - int(0.005 * fs)\n    base = np.median(med[b0:b1], axis=0)\n    stj = np.mean(med[j:j + int(0.004 * fs) + 1], axis=0)\n    st60 = med[min(len(med) - 1, j + int(0.06 * fs))]\n    st_mm = (stj - base) * 10.0\n    umb = umbrales(edad, sexo)\n    elev = {l: bool(st_mm[i] >= umb[l]) for i, l in enumerate(LEADS) if l != \'aVR\'}\n    elev[\'aVR\'] = False\n    cumple, territorios = criterio(elev)\n    ruido_base = float(np.max(np.std(med[b0:b1], axis=0)) * 10)\n    return {\n        \'med\': med, \'fs\': fs, \'onset\': onset, \'j\': j, \'r\': int(0.30 * fs),\n        \'base\': base, \'st_mm\': {l: float(round(v, 2)) for l, v in zip(LEADS, st_mm)},\n        \'st60_mm\': {l: float(round(v, 2)) for l, v in zip(LEADS, (st60 - base) * 10)},\n        \'umbral\': umb, \'elevadas\': [l for l in LEADS if elev[l]],\n        \'cumple\': cumple, \'territorios\': territorios, \'n_latidos\': n,\n        \'qrs_ms\': round((j - onset) * 1000 / fs), \'ruido_base_mm\': round(ruido_base, 2),\n        \'senal\': y,\n    }\n\n\ndef umbrales(edad=None, sexo=None):\n    """Umbral (mm) por derivación según la Quinta Definición Universal (2026)."""\n    u = {l: 1.0 for l in LEADS}\n    if sexo == \'M\':\n        v = 1.5\n    elif sexo == \'H\':\n        v = 2.0 if (edad is None or edad >= 40) else 2.5\n    else:\n        v = 2.0\n    u[\'V2\'] = u[\'V3\'] = v\n    return u\n\n\ndef criterio(elevadas):\n    """¿Hay elevación en >= 2 derivaciones contiguas? Devuelve (bool, territorios)."""\n    if isinstance(elevadas, (list, set, tuple)):\n        s = set(elevadas)\n    else:\n        s = {l for l, v in elevadas.items() if v}\n    s.discard(\'aVR\')\n    terr = [t for t, ls in TERRITORIOS.items() if len(s & set(ls)) >= 2]\n    # Contigüidad precordial a través del límite anterior/lateral (V4-V5)\n    if {\'V4\', \'V5\'} <= s and not terr:\n        terr = [\'Anterior\', \'Lateral\']\n    return (len(terr) > 0), terr\n\n\ndef sexo_ptbxl(valor):\n    """PTB-XL codifica sexo como 0 = hombre, 1 = mujer (verificar en la documentación)."""\n    try:\n        return \'H\' if int(valor) == 0 else \'M\'\n    except (TypeError, ValueError):\n        return None\n\n\ndef edad_ptbxl(valor):\n    """PTB-XL anonimiza edades > 89 años como 300."""\n    try:\n        e = float(valor)\n    except (TypeError, ValueError):\n        return None\n    if np.isnan(e):\n        return None\n    return 90.0 if e >= 300 else e\n\n\n# ----------------------------------------------------------------------------\n# Gráficas en papel de ECG (25 mm/s, 10 mm/mV, 1 mm = 0.04 s = 0.1 mV)\n# ----------------------------------------------------------------------------\ndef _papel(ax, x0, x1, y0, y1):\n    menores, mayores = [], []\n    for k, x in enumerate(np.arange(x0, x1 + 1e-9, 0.04)):\n        (mayores if k % 5 == 0 else menores).append([(x, y0), (x, y1)])\n    ys = np.arange(np.floor(y0 * 10) / 10, y1 + 1e-9, 0.1)\n    for y in ys:\n        k = int(round(y * 10))\n        (mayores if k % 5 == 0 else menores).append([(x0, y), (x1, y)])\n    ax.add_collection(LineCollection(menores, colors=_MINOR, linewidths=0.4, zorder=0))\n    ax.add_collection(LineCollection(mayores, colors=_MAJOR, linewidths=0.9, zorder=0))\n    ax.set_xlim(x0, x1)\n    ax.set_ylim(y0, y1)\n    ax.set_aspect(0.4)  # 1 mV (10 mm) mide 0.4 veces 1 s (25 mm)\n    ax.set_xticks([])\n    ax.set_yticks([])\n    for s in ax.spines.values():\n        s.set_visible(False)\n\n\ndef graficar_12(sig, fs, titulo=None, resaltar=None, ancho=16):\n    """Trazo estándar 3 x 4 (2.5 s por columna) + tira de ritmo DII."""\n    columnas = [[\'I\', \'II\', \'III\'], [\'aVR\', \'aVL\', \'aVF\'],\n                [\'V1\', \'V2\', \'V3\'], [\'V4\', \'V5\', \'V6\']]\n    sep = 3.0\n    alto = 4 * sep\n    fig, ax = plt.subplots(figsize=(ancho, ancho * alto * 0.4 / 10.4 + 0.4))\n    _papel(ax, -0.4, 10.0, -alto + 1.2, 1.2)\n    n = int(2.5 * fs)\n    t = np.arange(n) / fs\n    for c, col in enumerate(columnas):\n        for f, lead in enumerate(col):\n            y0 = -f * sep\n            i = LEADS.index(lead)\n            seg = sig[c * n:(c + 1) * n, i]\n            seg = seg - np.median(sig[:, i])\n            ax.plot(c * 2.5 + t, y0 + seg, color=_TRACE, lw=0.9)\n            color = \'#c0392b\' if resaltar and lead in resaltar else \'#1a1a1a\'\n            peso = \'bold\' if resaltar and lead in resaltar else \'normal\'\n            ax.text(c * 2.5 + 0.05, y0 + 0.75, lead, fontsize=11, color=color, weight=peso)\n            if c > 0:\n                ax.plot([c * 2.5, c * 2.5], [y0 - 0.3, y0 + 0.3], color=_TRACE, lw=1)\n    # Tira de ritmo\n    y0 = -3 * sep\n    ii = sig[:int(10 * fs), 1] - np.median(sig[:, 1])\n    ax.plot(np.arange(len(ii)) / fs, y0 + ii, color=_TRACE, lw=0.9)\n    ax.text(0.05, y0 + 0.75, \'II\', fontsize=11)\n    # Pulso de calibración 1 mV x 0.2 s\n    for f in range(4):\n        y0 = -f * sep\n        ax.plot([-0.35, -0.3, -0.3, -0.1, -0.1, -0.05],\n                [y0, y0, y0 + 1, y0 + 1, y0, y0], color=_TRACE, lw=1)\n    if titulo:\n        ax.set_title(titulo, fontsize=12, loc=\'left\')\n    fig.text(0.99, 0.01, \'25 mm/s · 10 mm/mV\', ha=\'right\', fontsize=8, color=\'#555\')\n    plt.tight_layout()\n    return fig\n\n\ndef graficar_latidos(an, derivaciones=None, mostrar_medida=True, columnas=6, ancho=16):\n    """Latido mediano de cada derivación con línea isoeléctrica, punto J y ST sombreado."""\n    derivaciones = derivaciones or LEADS\n    fs, med = an[\'fs\'], an[\'med\']\n    t = (np.arange(len(med)) - an[\'r\']) / fs\n    filas = int(np.ceil(len(derivaciones) / columnas))\n    fig, axes = plt.subplots(filas, columnas, figsize=(ancho, filas * 2.9))\n    axes = np.atleast_1d(axes).ravel()\n    for k, lead in enumerate(derivaciones):\n        ax = axes[k]\n        i = LEADS.index(lead)\n        y = med[:, i] - an[\'base\'][i]\n        _papel(ax, -0.20, 0.44, -1.2, 1.6)\n        ax.plot(t, y, color=_TRACE, lw=1.3)\n        ax.axhline(0, color=\'#2471a3\', ls=\'--\', lw=1)\n        tj = t[an[\'j\']]\n        if mostrar_medida:\n            sel = (t >= tj) & (t <= tj + 0.08)\n            st = an[\'st_mm\'][lead]\n            elev = lead in an[\'elevadas\']\n            ax.fill_between(t[sel], 0, y[sel], color=\'#e74c3c\' if elev else \'#95a5a6\', alpha=0.35)\n            ax.plot(tj, y[an[\'j\']], \'o\', color=\'#c0392b\', ms=5)\n            ax.set_title(f"{lead}: ST {st:+.1f} mm (umbral {an[\'umbral\'][lead]:.1f})",\n                         fontsize=10, color=\'#c0392b\' if elev else \'#333\')\n        else:\n            ax.set_title(lead, fontsize=10)\n    for ax in axes[len(derivaciones):]:\n        ax.axis(\'off\')\n    plt.tight_layout()\n    return fig\n\n\ndef graficar_anatomia(an, lead=\'V2\', ancho=5.5):\n    """Latido mediano anotado: onda P, segmento PR (línea de base), QRS, punto J, ST y onda T."""\n    fs, med = an[\'fs\'], an[\'med\']\n    i = LEADS.index(lead)\n    t = (np.arange(len(med)) - an[\'r\']) / fs\n    y = med[:, i] - an[\'base\'][i]\n    x0, x1 = -0.22, 0.46\n    y0 = float(np.clip(np.floor((y.min() - 0.2) * 2) / 2, -1.2, -0.5))\n    y1 = float(np.clip(np.ceil((y.max() + 0.4) * 2) / 2, 1.0, 1.8))\n    fig, ax = plt.subplots(figsize=(ancho, ancho * (y1 - y0) * 0.4 / (x1 - x0)))\n    _papel(ax, x0, x1, y0, y1)\n    ax.plot(t, y, color=_TRACE, lw=1.6)\n    ax.axhline(0, color=\'#2471a3\', ls=\'--\', lw=1.2)\n    to, tj = t[an[\'onset\']], t[an[\'j\']]\n    ax.axvspan(to - 0.025, to - 0.005, color=\'#2471a3\', alpha=0.15)\n    ax.axvspan(to, tj, color=\'#f1c40f\', alpha=0.15)\n    ax.plot(tj, y[an[\'j\']], \'o\', color=\'#c0392b\', ms=8)\n    ax.annotate(\'\', xy=(tj + 0.004, y[an[\'j\']]), xytext=(tj + 0.004, 0),\n                arrowprops=dict(arrowstyle=\'<->\', color=\'#c0392b\', lw=1.5))\n    ax.text(x0 + 0.01, -0.45, \'Segmento PR = línea de base\\n(línea azul punteada)\',\n            color=\'#2471a3\', fontsize=10, ha=\'left\')\n    ax.text((to + tj) / 2, y1 - 0.25, \'QRS\', ha=\'center\', fontsize=10, color=\'#7d6608\')\n    ax.text(tj + 0.015, y[an[\'j\']] + 0.15, f"Punto J\\nST = {an[\'st_mm\'][lead]:+.1f} mm",\n            color=\'#c0392b\', fontsize=10)\n    ax.set_title(f\'Anatomía del segmento ST — derivación {lead}\', loc=\'left\')\n    plt.tight_layout()\n    return fig\n',
            'ecg_st_ui.py': '# -*- coding: utf-8 -*-\n"""Banco de casos y entorno interactivo (ipywidgets) para la clase de elevación del ST."""\nimport io\nimport random\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport ipywidgets as W\nfrom IPython.display import display, clear_output, Markdown\n\nfrom ecg_st_core import (LEADS, TERRITORIOS, cargar_registro, analizar, graficar_12,\n                         graficar_latidos, criterio, umbrales, sexo_ptbxl, edad_ptbxl)\n\nEXCLUIR_CODIGOS = {\'CLBBB\', \'ILBBB\', \'CRBBB\', \'LVH\', \'WPW\', \'PACE\'}\nCOLUMNAS_RUIDO = [\'baseline_drift\', \'static_noise\', \'burst_noise\',\n                  \'electrodes_problems\', \'pacemaker\']\n\n\n# ----------------------------------------------------------------------------\n# Banco de casos\n# ----------------------------------------------------------------------------\ndef candidatos(db, scp):\n    """Selecciona candidatos a \'elevación del ST\' y a \'control normal\' en PTB-XL.\n\n    La etiqueta de PTB-XL solo sirve para ENRIQUECER la búsqueda: la clasificación\n    final la hace la medición en el punto J y, después, el instructor.\n    """\n    desc = scp[\'description\'].astype(str).str.lower()\n    # STE_ = elevación del ST inespecífica (la única declaración de PTB-XL que describe elevación).\n    # Clase MI = infarto (con frecuencia antiguo). INJ* = \'subendocardial injury\' = DEPRESIÓN del ST:\n    # se dejan al final; la medición en el punto J decide.\n    cod_lesion = {\'STE_\'} & set(scp.index)\n    cod_im = set(scp.index[scp.get(\'diagnostic_class\', pd.Series(index=scp.index)).astype(str) == \'MI\'])\n    cod_im |= set(scp.index[desc.str.contains(\'injury\')])\n\n    limpio = pd.Series(True, index=db.index)\n    for c in COLUMNAS_RUIDO:\n        if c in db.columns:\n            limpio &= db[c].isna()\n    sin_excluir = db[\'scp_codes\'].apply(lambda d: not (set(d) & EXCLUIR_CODIGOS))\n    base = db[limpio & sin_excluir]\n\n    lesion = base[base[\'scp_codes\'].apply(lambda d: bool(set(d) & cod_lesion))]\n    im = base[base[\'scp_codes\'].apply(lambda d: bool(set(d) & cod_im)) & ~base.index.isin(lesion.index)]\n    normal = base[base[\'scp_codes\'].apply(\n        lambda d: d.get(\'NORM\', 0) >= 100 and set(d) <= {\'NORM\', \'SR\'})]\n    return lesion, im, normal\n\n\ndef _fila(ecg_id, r, an, grupo):\n    fila = {\'ecg_id\': int(ecg_id), \'filename_hr\': r[\'filename_hr\'],\n            \'edad\': edad_ptbxl(r.get(\'age\')), \'sexo\': sexo_ptbxl(r.get(\'sex\')),\n            \'grupo\': grupo, \'ajuste_J_ms\': 0, \'verificado\': False, \'comentario\': \'\',\n            \'derivaciones_elevadas\': \' \'.join(an[\'elevadas\']),\n            \'territorios\': \' \'.join(an[\'territorios\']), \'qrs_ms\': an[\'qrs_ms\'],\n            \'ruido_base_mm\': an[\'ruido_base_mm\']}\n    fila.update({f\'st_{l}\': an[\'st_mm\'][l] for l in LEADS})\n    return fila\n\n\ndef construir_banco(db, scp, n_ste=30, n_ctrl=30, semilla=2026, max_revisar=400, verbose=True):\n    """Banco PROVISIONAL: casos donde la medición confirma >= 2 derivaciones contiguas\n    elevadas y controles normales sin ninguna derivación por arriba de su umbral."""\n    lesion, im, normal = candidatos(db, scp)\n    rng = np.random.default_rng(semilla)\n    pool_ste = list(rng.permutation(lesion.index)) + list(rng.permutation(im.index))\n    pool_ctrl = list(rng.permutation(normal.index))\n    filas, revisados = [], 0\n    for grupo, pool, meta in ((\'STE\', pool_ste, n_ste), (\'Control\', pool_ctrl, n_ctrl)):\n        k = 0\n        for ecg_id in pool:\n            if k >= meta or revisados >= max_revisar:\n                break\n            revisados += 1\n            r = db.loc[ecg_id]\n            try:\n                sig, fs = cargar_registro(r[\'filename_hr\'])\n                an = analizar(sig, fs, edad_ptbxl(r.get(\'age\')), sexo_ptbxl(r.get(\'sex\')))\n            except Exception as e:  # registro dañado o error de red\n                if verbose:\n                    print(\'  omitido\', ecg_id, type(e).__name__)\n                continue\n            if an is None or an[\'ruido_base_mm\'] > 0.3 or not (60 <= an[\'qrs_ms\'] <= 120):\n                continue\n            if grupo == \'STE\' and not an[\'cumple\']:\n                continue\n            if grupo == \'Control\':\n                margen = {l: an[\'umbral\'][l] * 0.6 for l in LEADS}\n                if any(an[\'st_mm\'][l] >= margen[l] for l in LEADS if l != \'aVR\'):\n                    continue\n            filas.append(_fila(ecg_id, r, an, grupo))\n            k += 1\n            if verbose:\n                print(f\'{grupo:8s} {k:3d}/{meta}  ecg_id={ecg_id}  {an["territorios"]}\')\n    banco = pd.DataFrame(filas)\n    if verbose:\n        print(f\'\\nRegistros revisados: {revisados}. Banco: {len(banco)} casos.\')\n    return banco\n\n\ndef cargar_banco(url=None, db=None, scp=None, n=12, subir=False):\n    """Banco verificado (CSV del instructor: enlace o archivo subido) o, si no hay, uno provisional pequeño."""\n    if subir:\n        from google.colab import files\n        up = files.upload()\n        url = io.BytesIO(next(iter(up.values())))\n    if url is not None and url != \'\':\n        banco = pd.read_csv(url)\n        if \'verificado\' in banco.columns and banco[\'verificado\'].any():\n            banco = banco[banco[\'verificado\']]\n        banco = banco[banco[\'grupo\'].isin([\'STE\', \'Control\'])]\n        display(Markdown(f\'**Banco verificado por el instructor:** {len(banco)} casos.\'))\n        return banco.reset_index(drop=True)\n    display(Markdown(\'⚠️ **Banco PROVISIONAL** (medición automática, sin verificación del instructor). \'\n                     \'Construyendo... tarda 1–3 minutos.\'))\n    banco = construir_banco(db, scp, n_ste=n, n_ctrl=n, verbose=False)\n    display(Markdown(f\'Listo: {len(banco)} casos.\'))\n    return banco\n\n\ndef analizar_caso(fila):\n    sig, fs = cargar_registro(fila[\'filename_hr\'])\n    edad = None if pd.isna(fila.get(\'edad\')) else float(fila[\'edad\'])\n    sexo = fila.get(\'sexo\') if isinstance(fila.get(\'sexo\'), str) else None\n    return analizar(sig, fs, edad, sexo, ajuste_J_ms=float(fila.get(\'ajuste_J_ms\', 0) or 0))\n\n\ndef _paciente(fila):\n    sexo = {\'H\': \'Hombre\', \'M\': \'Mujer\'}.get(fila.get(\'sexo\'), \'Sexo no registrado\')\n    edad = \'\' if pd.isna(fila.get(\'edad\')) else f", {int(fila[\'edad\'])} años"\n    return sexo + edad\n\n\n# ----------------------------------------------------------------------------\n# Revisión del instructor\n# ----------------------------------------------------------------------------\nclass RevisionInstructor:\n    """Recorre el banco provisional: aceptar / rechazar / corregir el punto J."""\n\n    def __init__(self, banco):\n        self.b = banco.copy().reset_index(drop=True)\n        self.i = 0\n        self.out = W.Output()\n        self.aj = W.IntSlider(value=0, min=-40, max=40, step=2, description=\'Ajuste J (ms)\',\n                              continuous_update=False, layout=W.Layout(width=\'420px\'))\n        self.grupo = W.ToggleButtons(options=[\'STE\', \'Control\', \'Excluir\'], description=\'Clasificación\')\n        self.com = W.Text(description=\'Comentario\', layout=W.Layout(width=\'600px\'))\n        b_ok = W.Button(description=\'Guardar y siguiente\', button_style=\'success\')\n        b_ant = W.Button(description=\'◀ Anterior\')\n        b_exp = W.Button(description=\'Exportar CSV\', button_style=\'info\')\n        self.estado = W.HTML()\n        b_ok.on_click(self._guardar)\n        b_ant.on_click(lambda _: self._ir(self.i - 1))\n        b_exp.on_click(self._exportar)\n        self.aj.observe(lambda _: self._mostrar(), names=\'value\')\n        display(W.VBox([self.estado, W.HBox([self.grupo, self.aj]), self.com,\n                        W.HBox([b_ant, b_ok, b_exp]), self.out]))\n        self._ir(0)\n\n    def _ir(self, i):\n        self.i = max(0, min(len(self.b) - 1, i))\n        f = self.b.loc[self.i]\n        self.aj.unobserve_all()\n        self.aj.value = int(f[\'ajuste_J_ms\'])\n        self.aj.observe(lambda _: self._mostrar(), names=\'value\')\n        self.grupo.value = f[\'grupo\'] if f[\'grupo\'] in (\'STE\', \'Control\') else \'Excluir\'\n        self.com.value = str(f[\'comentario\']) if isinstance(f[\'comentario\'], str) else \'\'\n        self._mostrar()\n\n    def _mostrar(self):\n        f = self.b.loc[self.i].copy()\n        f[\'ajuste_J_ms\'] = self.aj.value\n        an = analizar_caso(f)\n        n_ver = int(self.b[\'verificado\'].sum())\n        self.estado.value = (f"<b>Caso {self.i + 1}/{len(self.b)}</b> · ecg_id {f[\'ecg_id\']} · "\n                             f"{_paciente(f)} · propuesta automática: <b>{self.b.loc[self.i, \'grupo\']}</b> · "\n                             f"verificados: {n_ver}")\n        with self.out:\n            clear_output(wait=True)\n            graficar_12(an[\'senal\'], an[\'fs\'], f"ecg_id {f[\'ecg_id\']} — {_paciente(f)}",\n                        resaltar=an[\'elevadas\'])\n            plt.show()\n            graficar_latidos(an)\n            plt.show()\n            print(\'Cumple criterio:\', an[\'cumple\'], \'| Territorios:\', an[\'territorios\'],\n                  \'| QRS\', an[\'qrs_ms\'], \'ms | ruido basal\', an[\'ruido_base_mm\'], \'mm\')\n        self._an = an\n\n    def _guardar(self, _):\n        an = self._an\n        self.b.loc[self.i, \'ajuste_J_ms\'] = self.aj.value\n        self.b.loc[self.i, \'grupo\'] = self.grupo.value\n        self.b.loc[self.i, \'comentario\'] = self.com.value\n        self.b.loc[self.i, \'verificado\'] = self.grupo.value != \'Excluir\'\n        self.b.loc[self.i, \'derivaciones_elevadas\'] = \' \'.join(an[\'elevadas\'])\n        self.b.loc[self.i, \'territorios\'] = \' \'.join(an[\'territorios\'])\n        for l in LEADS:\n            self.b.loc[self.i, f\'st_{l}\'] = an[\'st_mm\'][l]\n        if self.grupo.value == \'STE\' and not an[\'cumple\']:\n            with self.out:\n                print(\'⚠️ Marcó STE pero la medición no cumple el criterio; revise el punto J.\')\n        self._ir(self.i + 1)\n\n    def _exportar(self, _):\n        nombre = \'casos_st_verificados.csv\'\n        self.b.to_csv(nombre, index=False)\n        with self.out:\n            print(f\'Guardado {nombre}: {int(self.b["verificado"].sum())} verificados de {len(self.b)}.\')\n        try:\n            from google.colab import files\n            files.download(nombre)\n        except Exception:\n            pass\n\n\n# ----------------------------------------------------------------------------\n# Módulo: medir el ST en una derivación\n# ----------------------------------------------------------------------------\nclass TallerMedicion:\n    """El alumno ubica el punto J y estima el desnivel; el entorno compara con la referencia."""\n\n    def __init__(self, banco):\n        self.ste = banco[banco[\'grupo\'] == \'STE\'].reset_index(drop=True)\n        self.out = W.Output()\n        self.caso = W.Dropdown(options=[(f\'Caso {k + 1}\', k) for k in range(len(self.ste))],\n                               description=\'ECG\')\n        self.lead = W.Dropdown(options=[l for l in LEADS if l != \'aVR\'], value=\'V2\',\n                               description=\'Derivación\')\n        self.j = W.IntSlider(value=60, min=0, max=160, step=2, description=\'Punto J (ms tras R)\',\n                             style={\'description_width\': \'initial\'}, continuous_update=False,\n                             layout=W.Layout(width=\'480px\'))\n        self.mm = W.FloatText(value=0.0, step=0.5, description=\'Tu medida (mm)\',\n                              style={\'description_width\': \'initial\'})\n        b = W.Button(description=\'Comprobar\', button_style=\'primary\')\n        b.on_click(self._comprobar)\n        for w in (self.caso, self.lead):\n            w.observe(lambda _: self._cargar(), names=\'value\')\n        self.j.observe(lambda _: self._dibujar(False), names=\'value\')\n        display(W.VBox([W.HBox([self.caso, self.lead]), self.j, W.HBox([self.mm, b]), self.out]))\n        self._cargar()\n\n    def _cargar(self):\n        self.an = analizar_caso(self.ste.loc[self.caso.value])\n        # Sugerir la derivación más elevada al cambiar de caso\n        self._dibujar(False)\n\n    def _dibujar(self, revelar):\n        an, lead = self.an, self.lead.value\n        i = LEADS.index(lead)\n        fs, med = an[\'fs\'], an[\'med\']\n        t = (np.arange(len(med)) - an[\'r\']) / fs * 1000\n        y = (med[:, i] - an[\'base\'][i]) * 10  # en mm\n        from ecg_st_core import _papel\n        with self.out:\n            clear_output(wait=True)\n            fig, ax = plt.subplots(figsize=(7, 5.2))\n            _papel(ax, -0.22, 0.46, -1.2, 1.8)\n            ax.plot(t / 1000, y / 10, color=\'#111\', lw=1.6)\n            ax.axhline(0, color=\'#2471a3\', ls=\'--\', lw=1)\n            jj = an[\'r\'] + int(self.j.value * fs / 1000)\n            ax.axvline(t[jj] / 1000, color=\'#8e44ad\', lw=1.2)\n            ax.plot(t[jj] / 1000, y[jj] / 10, \'s\', color=\'#8e44ad\', ms=7, label=\'Tu punto J\')\n            if revelar:\n                ax.axvline(t[an[\'j\']] / 1000, color=\'#c0392b\', ls=\':\', lw=1.5)\n                ax.plot(t[an[\'j\']] / 1000, y[an[\'j\']] / 10, \'o\', color=\'#c0392b\', ms=8,\n                        label=\'Punto J de referencia\')\n            ax.legend(loc=\'upper right\', fontsize=9)\n            ax.set_title(f\'{lead} — {_paciente(self.ste.loc[self.caso.value])}\', loc=\'left\')\n            plt.show()\n            if revelar:\n                ref = an[\'st_mm\'][lead]\n                tuyo = self.mm.value\n                umb = an[\'umbral\'][lead]\n                dif_j = (jj - an[\'j\']) * 1000 / fs\n                ok_mm = abs(tuyo - ref) <= 0.5\n                ok_j = abs(dif_j) <= 20\n                display(Markdown(\n                    f"- **Punto J:** {\'✅\' if ok_j else \'❌\'} el tuyo está a {dif_j:+.0f} ms del de referencia "\n                    f"(tolerancia ±20 ms = medio cuadrito).\\n"\n                    f"- **Desnivel:** {\'✅\' if ok_mm else \'❌\'} tú: {tuyo:+.1f} mm · referencia: {ref:+.1f} mm "\n                    f"(tolerancia ±0.5 mm).\\n"\n                    f"- **Umbral en {lead}** para este paciente: {umb:.1f} mm → "\n                    f"{\'**supera** el umbral\' if ref >= umb else \'no supera el umbral\'}."))\n\n    def _comprobar(self, _):\n        self._dibujar(True)\n\n\n# ----------------------------------------------------------------------------\n# Módulo: derivaciones contiguas\n# ----------------------------------------------------------------------------\nEJERCICIOS_CONTIGUAS = [\n    ([\'II\', \'aVF\'], \'Sí — Inferior\'),\n    ([\'V2\', \'V3\'], \'Sí — Anterior\'),\n    ([\'III\', \'V2\'], \'No — no son contiguas\'),\n    ([\'I\', \'aVL\'], \'Sí — Lateral\'),\n    ([\'aVF\'], \'No — solo una derivación\'),\n    ([\'V5\', \'V6\'], \'Sí — Lateral\'),\n    ([\'II\', \'V4\'], \'No — no son contiguas\'),\n]\n\n\ndef ejercicios_contiguas():\n    opciones = [\'Sí — Inferior\', \'Sí — Anterior\', \'Sí — Lateral\',\n                \'No — no son contiguas\', \'No — solo una derivación\']\n    cajas = []\n    for k, (ls, resp) in enumerate(EJERCICIOS_CONTIGUAS):\n        r = W.RadioButtons(options=opciones, value=None, layout=W.Layout(width=\'260px\'))\n        fb = W.HTML()\n\n        def _cb(cambio, resp=resp, fb=fb):\n            if cambio[\'new\'] == resp:\n                fb.value = \'✅ Correcto\'\n            else:\n                fb.value = f\'❌ Respuesta: <b>{resp}</b>\'\n        r.observe(_cb, names=\'value\')\n        cajas.append(W.VBox([W.HTML(f\'<b>{k + 1}.</b> ST elevado (por arriba del umbral) en: \'\n                                    f\'<b>{", ".join(ls)}</b>\'), r, fb],\n                            layout=W.Layout(border=\'1px solid #ddd\', padding=\'6px\', margin=\'4px\')))\n    display(W.GridBox(cajas, layout=W.Layout(grid_template_columns=\'repeat(3, 330px)\')))\n\n\n# ----------------------------------------------------------------------------\n# Módulo: práctica deliberada\n# ----------------------------------------------------------------------------\nVINETAS = [\n    \'dolor opresivo retroesternal de {h} h de evolución, irradiado al brazo izquierdo, con diaforesis\',\n    \'dolor torácico opresivo de {h} h que inició en reposo\',\n    \'malestar epigástrico, náusea y disnea de {h} h de evolución\',\n    \'dolor en el pecho de {h} h que no cede con el reposo, acompañado de ansiedad\',\n    \'sensación de ardor en el pecho de {h} h, irradiada a mandíbula\',\n]\nDEC_SI = \'Sí: compatible con IAM CEST → activar Código Infarto\'\nDEC_NO = \'No: no cumple criterio → diagnóstico diferencial\'\n\n\nclass Practica:\n    """Casos aleatorios (50 % con elevación, 50 % normales) dentro de una viñeta de Código Infarto,\n    con cronómetro de decisión y retroalimentación inmediata. Las viñetas son ficticias."""\n\n    def __init__(self, banco, semilla=None):\n        self.banco = banco.reset_index(drop=True)\n        self.rng = random.Random(semilla)\n        self.vistos = []\n        self.marcador = {\'n\': 0, \'decision\': 0, \'territorio\': 0, \'meta\': 0, \'tiempos\': []}\n        self.out_ecg, self.out_fb = W.Output(), W.Output()\n        self.vineta = W.HTML()\n        self.meta = W.ToggleButtons(options=[\'Sí, ≤ 10 min\', \'No, > 10 min\'], value=None,\n                                    description=\'1) ¿Se cumplió la meta de ECG en ≤ 10 min?\',\n                                    style={\'description_width\': \'initial\'})\n        self.dec = W.RadioButtons(options=[DEC_SI, DEC_NO], value=None,\n                                  description=\'2) ¿Trazo compatible con IAM CEST?\',\n                                  style={\'description_width\': \'initial\'},\n                                  layout=W.Layout(width=\'720px\'))\n        self.ders = {l: W.Checkbox(value=False, description=l, indent=False,\n                                   layout=W.Layout(width=\'70px\'))\n                     for l in LEADS if l != \'aVR\'}\n        self.terr = {t: W.Checkbox(value=False, description=t, indent=False,\n                                   layout=W.Layout(width=\'110px\')) for t in TERRITORIOS}\n        b_rev = W.Button(description=\'Revisar mi respuesta\', button_style=\'primary\')\n        b_sig = W.Button(description=\'Siguiente caso ▶\', button_style=\'success\')\n        b_rev.on_click(self._revisar)\n        b_sig.on_click(lambda _: self._nuevo())\n        self.score = W.HTML()\n        panel = W.VBox([\n            self.score, self.vineta, self.meta, self.out_ecg, self.dec,\n            W.HTML(\'3) Derivaciones con ST elevado (marque las que superan su umbral):\'),\n            W.HBox(list(self.ders.values())),\n            W.HTML(\'4) Territorio:\'), W.HBox(list(self.terr.values())),\n            W.HBox([b_rev, b_sig]), self.out_fb])\n        display(panel)\n        self._nuevo()\n\n    def _elegir(self):\n        grupo = \'STE\' if self.rng.random() < 0.5 else \'Control\'\n        pool = [k for k in self.banco.index[self.banco[\'grupo\'] == grupo] if k not in self.vistos]\n        if not pool:\n            pool = [k for k in self.banco.index if k not in self.vistos]\n        if not pool:\n            self.vistos = []\n            pool = list(self.banco.index)\n        k = self.rng.choice(pool)\n        self.vistos.append(k)\n        return k\n\n    def _nuevo(self):\n        self.k = self._elegir()\n        self.fila = self.banco.loc[self.k]\n        self.an = analizar_caso(self.fila)\n        self.revisado = False\n        self.dec.value = None\n        self.meta.value = None\n        h = self.rng.choice([1, 2, 3, 4, 6, 8, 10])\n        self.min_ecg = self.rng.choice([3, 4, 6, 7, 9, 10, 12, 15, 18])\n        texto = self.rng.choice(VINETAS).format(h=h)\n        self.vineta.value = (\n            f"<div style=\'border-left:4px solid #6a1b9a;padding:6px 10px;background:#f6effa;color:#222\'>"\n            f"<b>Urgencias · Código Infarto.</b> {_paciente(self.fila)}. Llega por {texto}. "\n            f"El ECG de 12 derivaciones se tomó <b>{self.min_ecg} min</b> después de su llegada. "\n            f"<i>(Viñeta ficticia con fines docentes.)</i></div>")\n        for c in list(self.ders.values()) + list(self.terr.values()):\n            c.value = False\n        with self.out_fb:\n            clear_output()\n        with self.out_ecg:\n            clear_output(wait=True)\n            n = self.marcador[\'n\'] + 1\n            graficar_12(self.an[\'senal\'], self.an[\'fs\'], f\'Caso {n} — {_paciente(self.fila)}\')\n            plt.show()\n        self._marcador()\n        self.t0 = time.time()\n\n    def _marcador(self):\n        m = self.marcador\n        if m[\'n\']:\n            self.score.value = (f"<b>Casos revisados:</b> {m[\'n\']} · <b>Decisión correcta:</b> "\n                                f"{m[\'decision\']}/{m[\'n\']} ({100 * m[\'decision\'] / m[\'n\']:.0f} %) · "\n                                f"<b>Territorio correcto:</b> {m[\'territorio\']}/{m[\'n\']} · "\n                                f"<b>Meta ≤ 10 min:</b> {m[\'meta\']}/{m[\'n\']} · "\n                                f"<b>Tiempo mediano para decidir:</b> {np.median(m[\'tiempos\']):.0f} s")\n        else:\n            self.score.value = \'<b>Práctica:</b> lea el ECG y responda.\'\n\n    def _revisar(self, _):\n        if self.dec.value is None or self.meta.value is None:\n            with self.out_fb:\n                clear_output()\n                print(\'Responda primero las preguntas 1 (meta de 10 min) y 2 (¿compatible con IAM CEST?).\')\n            return\n        an, fila = self.an, self.fila\n        verdad = fila[\'grupo\'] == \'STE\'\n        ref_ders = set(an[\'elevadas\']) - {\'aVR\'}\n        ref_terr = set(an[\'territorios\']) if verdad else set()\n        suyo = self.dec.value == DEC_SI\n        meta_ok = self.min_ecg <= 10\n        ok_meta = (self.meta.value.startswith(\'Sí\')) == meta_ok\n        sus_ders = {l for l, c in self.ders.items() if c.value}\n        sus_terr = {t for t, c in self.terr.items() if c.value}\n        ok_dec = suyo == verdad\n        ok_terr = sus_terr == ref_terr\n        if not self.revisado:\n            self.marcador[\'n\'] += 1\n            self.marcador[\'decision\'] += int(ok_dec)\n            self.marcador[\'territorio\'] += int(ok_terr)\n            self.marcador[\'meta\'] += int(ok_meta)\n            self.marcador[\'tiempos\'].append(time.time() - self.t0)\n            self.revisado = True\n        self._marcador()\n        aciertos = sorted(sus_ders & ref_ders, key=LEADS.index)\n        omitidas = sorted(ref_ders - sus_ders, key=LEADS.index)\n        sobran = sorted(sus_ders - ref_ders, key=LEADS.index)\n        u = an[\'umbral\']\n        with self.out_fb:\n            clear_output(wait=True)\n            seg = self.marcador[\'tiempos\'][-1]\n            accion = (\'**Activar Código Infarto** y establecer el "tiempo 0"\' if verdad\n                      else \'No activar por ECG; continuar **diagnóstico diferencial** (repetir ECG si persiste el dolor)\')\n            display(Markdown(\n                f"### {\'✅\' if ok_dec else \'❌\'} Este ECG **{\'sí\' if verdad else \'no\'}** cumple el criterio "\n                f"de elevación del ST → {accion}\\n"\n                f"- Meta de ECG ≤ 10 min: {\'✅\' if ok_meta else \'❌\'} el ECG se tomó a los {self.min_ecg} min "\n                f"({\'meta cumplida\' if meta_ok else \'meta NO cumplida\'}). · Su tiempo de decisión: {seg:.0f} s\\n"\n                f"- Umbrales para este paciente ({_paciente(fila)}): V2–V3 ≥ {u[\'V2\']:.1f} mm; "\n                f"resto ≥ 1.0 mm, en ≥ 2 derivaciones contiguas.\\n"\n                f"- Derivaciones con ST elevado (referencia): **{\', \'.join(sorted(ref_ders, key=LEADS.index)) or \'ninguna\'}**\\n"\n                f"- Bien identificadas: {\', \'.join(aciertos) or \'—\'} · Omitidas: {\', \'.join(omitidas) or \'—\'} · "\n                f"Marcadas de más: {\', \'.join(sobran) or \'—\'}\\n"\n                f"- Territorio: {\'✅\' if ok_terr else \'❌\'} referencia: **{\', \'.join(sorted(ref_terr)) or \'ninguno\'}**"))\n            graficar_12(an[\'senal\'], an[\'fs\'], \'Derivaciones con ST elevado en rojo\',\n                        resaltar=ref_ders)\n            plt.show()\n            display(Markdown(\'**Latido mediano por derivación** (línea azul = segmento PR; punto rojo = \'\n                             \'punto J; sombra = desnivel del ST):\'))\n            graficar_latidos(an)\n            plt.show()\n\n\n# ----------------------------------------------------------------------------\n# Versión simple para alumnos: ejemplos fijos que se buscan solos\n# ----------------------------------------------------------------------------\ndef buscar_ejemplos(db, scp, max_revisar=150, semilla=2026):\n    """Un ECG anterior, uno inferior, uno lateral y uno normal, elegidos automáticamente."""\n    lesion, im, normal = candidatos(db, scp)\n    rng = np.random.default_rng(semilla)\n    pool = list(rng.permutation(lesion.index)) + list(rng.permutation(im.index))\n    buscados = {\'Anterior\': None, \'Inferior\': None, \'Lateral\': None}\n    vistos = 0\n    for ecg_id in pool:\n        if all(buscados.values()) or vistos >= max_revisar:\n            break\n        vistos += 1\n        r = db.loc[ecg_id]\n        try:\n            sig, fs = cargar_registro(r[\'filename_hr\'])\n            an = analizar(sig, fs, edad_ptbxl(r.get(\'age\')), sexo_ptbxl(r.get(\'sex\')))\n        except Exception:\n            continue\n        if an is None or not an[\'cumple\'] or an[\'ruido_base_mm\'] > 0.3:\n            continue\n        for t in an[\'territorios\']:\n            if buscados.get(t) is None and len(an[\'territorios\']) == 1:\n                buscados[t] = (r, an)\n    ejemplos = [(t, v[0], v[1]) for t, v in buscados.items() if v]\n    normales = []\n    for ecg_id in rng.permutation(normal.index)[:40]:\n        r = db.loc[ecg_id]\n        try:\n            sig, fs = cargar_registro(r[\'filename_hr\'])\n            an = analizar(sig, fs, edad_ptbxl(r.get(\'age\')), sexo_ptbxl(r.get(\'sex\')))\n        except Exception:\n            continue\n        if an and not an[\'elevadas\'] and an[\'ruido_base_mm\'] <= 0.3:\n            normales.append((\'Normal\', r, an))\n        if len(normales) >= len(ejemplos):\n            break\n    mezcla = []\n    for k in range(max(len(ejemplos), len(normales))):\n        mezcla += ejemplos[k:k + 1] + normales[k:k + 1]\n    return mezcla\n\n\ndef mostrar_ejemplo(n, territorio, r, an):\n    paciente = _paciente({\'sexo\': sexo_ptbxl(r.get(\'sex\')), \'edad\': edad_ptbxl(r.get(\'age\'))})\n    graficar_12(an[\'senal\'], an[\'fs\'], f\'Ejemplo {n} — {paciente}\', resaltar=an[\'elevadas\'] or None)\n    plt.show()\n    graficar_latidos(an)\n    plt.show()\n    if an[\'elevadas\']:\n        det = \', \'.join(f"{l} ({an[\'st_mm\'][l]:+.1f} mm; umbral {an[\'umbral\'][l]:.1f})" for l in an[\'elevadas\'])\n        display(Markdown(\n            f"**Lectura:** ST elevado en {det}. Son **derivaciones contiguas** del territorio **{territorio.lower()}** "\n            f"→ **compatible con IAM CEST** → en ICAT³: confirmar y **activar**."))\n    else:\n        display(Markdown(\n            "**Lectura:** en ninguna derivación el punto J supera su umbral → **no cumple** criterio de elevación "\n            "del ST. Si el paciente tiene síntomas, repita el ECG."))\n'}
for _n, _s in _FUENTES.items():
    with open(_n, 'w', encoding='utf-8') as _f:
        _f.write(_s)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import ecg_st_core, ecg_st_ui
importlib.reload(ecg_st_core); importlib.reload(ecg_st_ui)
from ecg_st_core import *
from ecg_st_ui import *
if os.environ.get('ECG_ST_PRUEBA'):  # solo para pruebas del autor sin internet
    sys.path.insert(0, os.environ['ECG_ST_PRUEBA'])
    import prueba_mock; prueba_mock.parchar(globals())
print('Entorno listo.')

In [ ]:
#@title ⏳ Cargando ECG reales (1–2 minutos)
db, scp = cargar_metadatos()
ejemplos = buscar_ejemplos(db, scp)
print(f'Listo: {len(ejemplos)} ECG de ejemplo.')

## 1. Las partes de la línea

Cada latido del ECG tiene partes con nombre. Para el ST solo necesita tres:

1. **Segmento PR (línea de base):** el tramo plano entre la onda P y el QRS. Es el **cero** (línea azul punteada).
2. **Punto J:** donde **termina el QRS** y la línea cambia bruscamente de dirección (punto rojo).
3. **Segmento ST:** lo que sigue al punto J. Si el punto J queda **por arriba** de la línea de base, hay
   **elevación del ST**. Se mide en milímetros: **1 cuadrito = 1 mm**.

In [ ]:
#@title 🔬 Un latido con sus partes señaladas
_t, _r, _an = ejemplos[0]
graficar_anatomia(_an, max(_an['elevadas'], key=lambda l: _an['st_mm'][l]) if _an['elevadas'] else 'II')
plt.show()

## 2. Cuándo la elevación cuenta (Quinta Definición Universal del Infarto, 2026)

Elevación **nueva** del ST en el **punto J**, en **2 o más derivaciones contiguas**:

| Derivaciones | Umbral |
|---|---|
| V2–V3, hombre ≥ 40 años | ≥ 2 mm |
| V2–V3, hombre < 40 años | ≥ 2.5 mm |
| V2–V3, mujer | ≥ 1.5 mm |
| Todas las demás | ≥ 1 mm |

**Derivaciones contiguas** (miran la misma zona del corazón): **inferior** II, III, aVF · **lateral** I, aVL, V5, V6 ·
**anterior** V1–V4.

## 3. Ejemplos con ECG reales

Arriba, el ECG completo: las derivaciones con ST elevado aparecen en **rojo**. Abajo, un latido de cada derivación
con la línea de base, el punto J y el ST sombreado. Al final, la lectura.

In [ ]:
#@title 🫀 Ejemplos
for _i, (_t, _r, _an) in enumerate(ejemplos, 1):
    display(Markdown(f'### Ejemplo {_i}: {"ECG normal" if _t == "Normal" else "territorio " + _t.lower()}'))
    mostrar_ejemplo(_i, _t, _r, _an)

## Conclusión

**Lo importante es el razonamiento clínico.** Los milímetros son una herramienta, no el objetivo. En un paciente
con síntomas, una elevación limítrofe o dudosa exige **alta sospecha**: repita el ECG, compárelo con trazos previos,
consulte de inmediato y **no espere a los biomarcadores**. Lo que salva miocardio es identificar, confirmar y
activar a tiempo.

### Referencias
1. Mills NL, Newby LK, Zaman S, et al. Fifth Universal Definition of Myocardial Infarction (2026). *Circulation*. 2026.
   doi:10.1161/CIR.0000000000001477
2. Byrne RA, et al. 2023 ESC Guidelines for the management of acute coronary syndromes. *Eur Heart J*.
   2023;44(38):3720–3826.
3. Instituto Mexicano del Seguro Social. Protocolo de Atención Integral de Código Infarto. Actualización. México: IMSS; octubre de 2025.
4. Programa Nacional para la Reducción de la Mortalidad por Infarto Agudo al Miocardio IAM_MX (PREMIA). Infarto agudo al miocardio. Grupo de Reperfusión en el Infarto Agudo al Miocardio, A.C.; 2017. Disponible en: calidad.salud.gob.mx/site/iam/docs/iam_01.pdf
5. Jerjes-Sánchez C, Martínez-Sánchez C, Borrayo-Sánchez G, Carrillo-Calvillo J, Juárez-Herrera U, Quintanilla-Gutiérrez J. Third national registry of acute coronary syndromes (RENASICA III). *Arch Cardiol Mex*. 2015;85:207–214.
6. Wagner P, et al. PTB-XL, a large publicly available electrocardiography dataset. *Sci Data*. 2020;7:154.
7. Goldberger AL, et al. PhysioBank, PhysioToolkit, and PhysioNet. *Circulation*. 2000;101:e215–e220.
8. Kligfield P, et al. Recommendations for the standardization and interpretation of the electrocardiogram, part I.
   *Circulation*. 2007;115:1306–1324.

---
**Autor:** Víctor Junior Godínez-Vázquez · Material docente · Texto CC BY 4.0 · Código MIT
**Datos:** PTB-XL v1.0.3 (PhysioNet, CC BY 4.0). Wagner P, et al. *Sci Data* 2020;7:154.
**Advertencia:** material exclusivamente docente; no usar para decisiones clínicas.
*Elaborado con asistencia de IA (Claude, Anthropic); el autor revisó el contenido y es responsable de él.*